# Sistem Prediksi DBD - Google Colab

Random Forest untuk prediksi Demam Berdarah Dengue (DBD)

**Cara pakai:** Jalankan semua cell dari atas ke bawah.

## 1. Clone & Install

In [ ]:
#@title Clone repo & install dependencies {{ display-mode: "form" }}
!git clone https://github.com/Suwardi87/prediksi-DBD.git /content/prediksi-DBD 2>/dev/null || true
%cd /content/prediksi-DBD/python_app
!pip install -q -r requirements.txt

## 2. Setup Database & Import Data

In [ ]:
#@title Setup SQLite + Auto-seed 163 pasien dari Excel {{ display-mode: "form" }}
import os, sys
os.environ['DATABASE_URL'] = 'sqlite:////content/prediksi-DBD/python_app/db.sqlite3'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
sys.path.insert(0, '/content/prediksi-DBD/python_app')

from app import create_app, db
from app.models import User, PasienDBD, KasusBulanan

# create_app() auto-seeds users + 163 patients from Excel if DB is empty
app = create_app()

with app.app_context():
    n_users = User.query.count()
    n_pasien = PasienDBD.query.count()
    n_kasus = KasusBulanan.query.count()

    # Verify data integrity
    okt = PasienDBD.query.filter_by(bulan='Oktober', tahun=2024).count()
    nov = PasienDBD.query.filter_by(bulan='November', tahun=2024).count()

    print(f'Users:        {n_users}')
    print(f'Pasien:       {n_pasien}')
    print(f'KasusBulanan: {n_kasus}')
    print(f'Oktober 2024: {okt} pasien (expected: 33)')
    print(f'November 2024: {nov} pasien (expected: 21)')
    print()
    if n_pasien == 163 and okt == 33 and nov == 21:
        print('DATA OK — Login: admin/admin123 atau petugas/petugas123')
    else:
        print('WARNING: Data mismatch! Check Excel file.')

## 3. Jalankan Aplikasi

In [ ]:
#@title Jalankan Flask + tampilkan via iframe {{ display-mode: "form" }}
import os, sys
os.environ['DATABASE_URL'] = 'sqlite:////content/prediksi-DBD/python_app/db.sqlite3'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
sys.path.insert(0, '/content/prediksi-DBD/python_app')

import threading, time
from app import create_app

app = create_app()

def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

# Kill any existing Flask on port 5000
!fuser -k 5000/tcp 2>/dev/null || true
time.sleep(1)

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()
time.sleep(3)

# Try pyngrok first, fallback to localtunnel
try:
    from pyngrok import ngrok
    public_url = ngrok.connect(5000)
    print(f'URL: {public_url}')
except Exception:
    try:
        !pip install -q localtunnel
        import subprocess
        lt = subprocess.Popen(['lt', '--port', '5000'],
                              stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        time.sleep(5)
        lt_url = lt.stdout.readline().decode().strip()
        print(f'URL: {lt_url}')
    except Exception:
        print('URL: http://localhost:5000 (gunakan port forwarding jika perlu)')

print('Login: admin / admin123')

## 4. Training Model (Opsional)

In [ ]:
#@title Training Random Forest {{ display-mode: "form" }}
import os, sys, json
os.environ['DATABASE_URL'] = 'sqlite:////content/prediksi-DBD/python_app/db.sqlite3'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
sys.path.insert(0, '/content/prediksi-DBD/python_app')

from app import create_app, db
from app.models import PasienDBD, KasusBulanan, ModelEvaluasi
from app.ml_model import prepare_training_data, train_model
from datetime import datetime

app = create_app()
with app.app_context():
    pasien_list = PasienDBD.query.all()
    kasus_dict = {}
    for kb in KasusBulanan.query.all():
        kasus_dict[(kb.bulan, kb.tahun)] = kb.jumlah_kasus

    df = prepare_training_data(pasien_list, kasus_dict)
    result = train_model(df, n_estimators=5, random_state=42)

    eval_record = ModelEvaluasi(
        tanggal_training=datetime.now(),
        accuracy=result['metrics']['accuracy'],
        precision_score=result['metrics']['precision_weighted'],
        recall_score=result['metrics']['recall_weighted'],
        f1_score=result['metrics']['f1_score_weighted'],
        mae=result['metrics']['mae'],
        rmse=result['metrics']['rmse'],
        r2_score=result['metrics']['r2_score'],
        n_estimators=5,
        confusion_matrix=json.dumps(result['confusion_matrix']),
        feature_importance=json.dumps(result['feature_importance']),
        model_path='models/random_forest_model.pkl')
    db.session.add(eval_record)
    db.session.commit()

    m = result['metrics']
    print(f'Accuracy:  {m["accuracy"]:.2%}')
    print(f'Precision: {m["precision_weighted"]:.2%}')
    print(f'Recall:    {m["recall_weighted"]:.2%}')
    print(f'F1 Score:  {m["f1_score_weighted"]:.2%}')
    print(f'MAE:       {m["mae"]:.4f}')
    print(f'RMSE:      {m["rmse"]:.4f}')
    print(f'R2 Score:  {m["r2_score"]:.4f}')
    print('Model berhasil! Buka aplikasi untuk melihat hasilnya.')